# Experiment 5.4.1 — direct WHAT base + constrained WHAT×WHEN residual

Analysis-only notebook for `direct_what_conjunction_residual_v2`. The base classifier reads the original frozen WHAT L2 spikes directly (`direct_what_wholecount`); the residual branch can add evidence only through a structural WHAT×WHEN conjunction.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if not (repo_root / 'notebooks').is_dir():
    repo_root = repo_root.parent
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_5_4_1_constrained_conjunction_residual' / 'direct_what_conjunction_residual_v2'
base_runs = pd.read_csv(root / 'base_runs.csv')
base_histories = pd.read_csv(root / 'base_histories.csv')
runs = pd.read_csv(root / 'runs.csv')
histories = pd.read_csv(root / 'histories.csv')
ablations = pd.read_csv(root / 'ablation_runs.csv')
activity = pd.read_csv(root / 'activity_runs.csv')
paired = pd.read_csv(root / 'paired_deltas.csv')
manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
manifest


## Direct WHAT base versus constrained residual

The primary paired quantity is residual-model test BA minus the direct whole-count readout of the exact same frozen WHAT spike trajectory.


In [ ]:
summary = pd.DataFrame({
    'base_condition': ['direct_what_wholecount'],
    'base_ba_mean': [base_runs['test_balanced_accuracy'].mean()],
    'base_ba_sd': [base_runs['test_balanced_accuracy'].std()],
    'residual_ba_mean': [runs['test_balanced_accuracy'].mean()],
    'residual_ba_sd': [runs['test_balanced_accuracy'].std()],
    'delta_vs_base_mean': [runs['delta_test_ba_vs_base'].mean()],
    'delta_vs_base_sd': [runs['delta_test_ba_vs_base'].std()],
    'wins_vs_base': [int((runs['delta_test_ba_vs_base'] > 0).sum())],
    'epoch0_selected': [int((runs['best_epoch'] == 0).sum())],
})
summary


In [ ]:
paired_summary = pd.DataFrame({
    'delta_ba_mean': [paired['delta_test_balanced_accuracy'].mean()],
    'delta_ba_sd': [paired['delta_test_balanced_accuracy'].std()],
    'wins': [int((paired['delta_test_balanced_accuracy'] > 0).sum())],
    'delta_macro_f1_mean': [paired['delta_test_macro_f1'].mean()],
})
paired_summary


## History/alignment attribution

`when_zero` and `residual_what_zero` are hard structural controls: residual evidence must be zero and final predictions must recover the frozen direct WHAT base. Ordered-vs-reset/shuffle/circular-shift tests whether any gain depends on history and temporal alignment.


In [ ]:
ablation_summary = (
    ablations.groupby('ablation', as_index=False)
    .agg(
        test_ba_mean=('test_balanced_accuracy', 'mean'),
        test_ba_sd=('test_balanced_accuracy', 'std'),
        residual_abs_mean=('residual_mean_abs_logit', 'mean'),
        residual_abs_max=('residual_abs_max', 'max'),
        conjunction_fr_mean=('conjunction_firing_fraction', 'mean'),
    )
    .sort_values('test_ba_mean', ascending=False)
)
ablation_summary


In [ ]:
ordered = ablations[ablations['ablation'] == 'ordered'].set_index('seed')
alignment_rows = []
for name in ('reset_when', 'when_shuffle', 'when_circular_shift'):
    control = ablations[ablations['ablation'] == name].groupby('seed')['test_balanced_accuracy'].mean()
    delta = ordered['test_balanced_accuracy'] - control
    alignment_rows.append({
        'ablation': name,
        'aligned_minus_ablation_ba_mean': delta.mean(),
        'aligned_minus_ablation_ba_sd': delta.std(),
        'aligned_wins': int((delta > 0).sum()),
    })
alignment = pd.DataFrame(alignment_rows)
alignment


In [ ]:
zero_checks = ablations[ablations['ablation'].isin(['when_zero', 'residual_what_zero'])]
zero_checks.groupby('ablation', as_index=False).agg(
    residual_abs_max=('residual_abs_max', 'max'),
    conjunction_fr_max=('conjunction_firing_fraction', 'max'),
    test_ba_mean=('test_balanced_accuracy', 'mean'),
)


## Base and residual training dynamics


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for seed, group in base_histories.groupby('seed'):
    ax.plot(group['epoch'], group['val_balanced_accuracy'], label=f'base seed {seed}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation balanced accuracy')
ax.set_title('Direct WHAT base validation BA')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for seed, group in histories.groupby('seed'):
    ax.plot(group['epoch'], group['val_balanced_accuracy'], label=f'residual seed {seed}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation balanced accuracy')
ax.set_title('Constrained WHAT×WHEN residual validation BA')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()


## Interpretation checklist

Support requires positive paired BA over `direct_what_wholecount`, ordered WHEN outperforming reset/shuffled/shifted controls, and exact zero residual under either missing conjunction side. If `epoch0_selected` is high, the correct conclusion is that the frozen WHEN representation did not add reliable incremental classification value beyond direct WHAT evidence.
